In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, r2_score

# -----------------------------
# 1️⃣ Custom transformer for daily data
# -----------------------------
class PreprocessDaily(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy()
        
        # Date features
        df['date'] = pd.to_datetime(df['date'])
        df['month'] = df['date'].dt.month
        df['day'] = df['date'].dt.day
        df['day_of_year'] = df['date'].dt.dayofyear
        df['day_of_week_code'] = df['date'].dt.dayofweek  # 0=Monday, 6=Sunday
        
        # Encode station
        df['station_code'] = df['station'].astype('category').cat.codes
        
        # Drop unused columns
        drop_cols = ['date', 'station', 'day_of_week', 'sunrise', 'sunset', 'hour']
        df = df.drop(columns=[c for c in drop_cols if c in df.columns])
        
        # Fill missing values
        df = df.fillna(0)
        
        return df

# -----------------------------
# 2️⃣ Load dataset
# -----------------------------
df = pd.read_csv('../datasets/raw/combined.csv')

# -----------------------------
# 3️⃣ Prepare target
# -----------------------------
target = 'overcrowding'

X = df.drop(columns=[
    'entries',
    'exits',
    'baseline_entries',
    'baseline_exits',
    target,
])
y = df[target]

# -----------------------------
# 4️⃣ Pipeline
# -----------------------------
pipeline = Pipeline([
    ('preprocess', PreprocessDaily()),
    ('rf', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
        max_samples=0.8
    ))
])

# -----------------------------
# 5️⃣ Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 6️⃣ Train model
# -----------------------------
import time
start = time.time()
pipeline.fit(X_train, y_train)
end = time.time()
print(f"Training time: {end - start:.2f} seconds")

# -----------------------------
# 7️⃣ Evaluate
# -----------------------------
preds = pipeline.predict(X_test)
print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

# -----------------------------
# 8️⃣ Feature importances
# -----------------------------
rf = pipeline.named_steps['rf']
preprocessed_X = pipeline.named_steps['preprocess'].transform(X_train)
feature_importances = pd.Series(rf.feature_importances_, index=preprocessed_X.columns)
print(feature_importances.sort_values(ascending=False).head(20))

Training time: 12.66 seconds
MAE: 9.839121674990286
R²: 0.24525622014713222
day_of_year         0.340979
station_code        0.275211
daylight_s          0.082188
day_of_week_code    0.061778
day                 0.050776
wind_dir            0.027695
sunshine_s          0.023805
app_temp_mean       0.023424
wind_gust_max       0.020810
app_temp_max        0.018444
app_temp_min        0.013310
wind_max            0.012070
precip_hours        0.009779
temp_min            0.009003
precip_mm           0.007807
rain_mm             0.006957
temp_max            0.006862
temp_mean           0.006220
month               0.001643
is_raining          0.001106
dtype: float64
